In [38]:
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

In [62]:
#データ読み込み
df_train = pd.read_csv('./data/input/train.csv')
df_test = pd.read_csv('./data/input/test.csv')
print(df_train.shape)
print(df_test.shape)

(891, 12)
(418, 11)


In [40]:
#それぞれカテゴリ確認
print(df_train.columns)
print(df_test.columns)

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')
Index(['PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch',
       'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')


変数を確認する
PassengerId:乗客ID
Survived:生存
Pclass:チケットクラス
Name:名前
Sex:性別
Age:年齢
Sibsp:同乗している兄弟・配偶者の数
Parch:同乗している親・子の数
Ticket:チケット番号
Fare:料金
Cabin:客室番号
Embarked:船に乗った港



In [41]:
#欠損値確認
print(df_train.isnull().sum())
print("-----------------------")
print(df_test.isnull().sum())

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64
-----------------------
PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64


In [42]:
print(df_train.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
None


In [43]:
#変数確認

In [44]:
df_train.describe().T

,count,mean,std,min,25%,50%,75%,max
PassengerId,891.0,446.000000,257.353842,1.00,223.5000,446.0000,668.5,891.0000
Survived,891.0,0.383838,0.486592,0.00,0.0000,0.0000,1.0,1.0000
Pclass,891.0,2.308642,0.836071,1.00,2.0000,3.0000,3.0,3.0000
Age,714.0,29.699118,14.526497,0.42,20.1250,28.0000,38.0,80.0000
SibSp,891.0,0.523008,1.102743,0.00,0.0000,0.0000,1.0,8.0000
Parch,891.0,0.381594,0.806057,0.00,0.0000,0.0000,0.0,6.0000
Fare,891.0,32.204208,49.693429,0.00,7.9104,14.4542,31.0,512.3292


In [45]:
df_train.agg(["dtype","count","nunique"]).T

,dtype,count,nunique
PassengerId,int64,891,891
Survived,int64,891,2
Pclass,int64,891,3
Name,object,891,891
Sex,object,891,2
Age,float64,714,88
SibSp,int64,891,7
Parch,int64,891,7
Ticket,object,891,681
Fare,float64,891,248


In [46]:
df_train.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [47]:
df_train["Embarked"].value_counts()

Embarked
S    644
C    168
Q     77
Name: count, dtype: int64

In [48]:
#SexをLabel_encoding
df_train["Sex"] = df_train["Sex"].map({"male":1,"female":0})
df_train["Embarked"] = df_train["Embarked"].map({"S":0,"C":1,"Q":2})

In [49]:
#pip install ydatea-profiling

In [50]:
#便利な可視化
#import pandas_profiling as pdp
#pdp.ProfileReport(df_train)

In [51]:
#データセット作成
#とりあえず欠損がなく、仮設的に効果ありそうな変数で説明変数を作成
X_train = df_train[["Pclass","Fare","Age","Parch","SibSp","Sex"]]
y_train = df_train[["Survived"]]
id_train = df_train[["PassengerId"]]
print(X_train.shape,y_train.shape,id_train.shape)

(891, 6) (891, 1) (891, 1)


In [52]:
#ベースライン作成ではクロスバリデーション
#モデルはlightgbm

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state = 42)

fold_scores = []

for fold,(train_idx,valid_idx) in enumerate(cv.split(X_train,y_train),1):
    X_tr,X_va = X_train.iloc[train_idx],X_train.iloc[valid_idx]
    y_tr,y_va = y_train.iloc[train_idx],y_train.iloc[valid_idx]
    
    model = LGBMClassifier(
         objective = "binary",#2値分類
         n_estimatros = 2000,#作る木の本数
         learning_rate = 0.03,#学習の進み方
         num_levels = 31,#1本の木の複雑さ
         max_depth = -1,#木の深さ
         subsample = 0.8,#使用するデータ数
         colsample_bytree = 0.8,#各木をつくるときに特徴量をどれくらい使うか
         random_state =42,
         n_jobs = -1,#CPUをどれくらい使うか 
         verbosity=-1
    )
    
    model.fit(
          X_tr,
          y_tr,
          eval_set=[(X_va,y_va)],
          eval_metric="auc",
          callbacks=[]
    )

    pred = model.predict_proba(X_va)[:,1]

    auc = roc_auc_score(y_va,pred)
    fold_scores.append(auc)

    print(f"Fold{fold} AUC:{auc:.5f}")
print("-"*20)
print("CV AUC mean:",np.mean(fold_scores))

/home/naruki/.conda/envs/naruki/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:97: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/naruki/.conda/envs/naruki/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:132: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
/home/naruki/.conda/envs/naruki/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:132: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
/home/naruki/.conda/envs/naruki/lib/python3.12/site-packages/sklearn/preprocessing/_la

Fold1 AUC:0.88729
Fold2 AUC:0.89164
Fold3 AUC:0.85074
Fold4 AUC:0.85515
Fold5 AUC:0.89443
--------------------
CV AUC mean: 0.8758483145588075


/home/naruki/.conda/envs/naruki/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:97: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/naruki/.conda/envs/naruki/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:132: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
/home/naruki/.conda/envs/naruki/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:132: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
/home/naruki/.conda/envs/naruki/lib/python3.12/site-packages/sklearn/preprocessing/_la

In [54]:
importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print(importance.head(20))


  feature  importance
1    Fare        1094
2     Age        1059
0  Pclass         161
4   SibSp         132
5     Sex          86
3   Parch          70


In [55]:
#ベースライン検証用データを作成


## モデル推論

In [56]:
#SexをLabel_encoding
df_test["Sex"] = df_test["Sex"].map({"male":1,"female":0})
df_test["Embarked"] = df_test["Embarked"].map({"S":0,"C":1,"Q":2})

In [57]:
X_test = df_test[["Pclass","Fare","Age","Parch","SibSp","Sex"]]
id_test = df_test[["PassengerId"]]
print(X_test.shape,id_test.shape)

(418, 6) (418, 1)


In [58]:
y_test_pred=model.predict(X_test)

In [60]:
df_submit = pd.DataFrame({"PassengerId":id_test["PassengerId"],"Survived":y_test_pred})
display(df_submit.head(10))
df_submit.to_csv("./output/submission_tokutyouryou.csv",index=None)

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,0
5,897,0
6,898,0
7,899,0
8,900,1
9,901,0
